# Experiment: Private Recipient Source Index Readiness Gate

Objective: classify whether an ignored private recipient source-index row can support a public source-bundle label without leaking raw candidate URLs, real identity, contacts, private replies, patient material, treatment, trial, purchase, import, or procurement scope.


In [ ]:
from __future__ import annotations

READY = "private_recipient_source_index_ready"
MISSING_SOURCE = "private_recipient_source_index_hold_missing_source"
MAPPING_INCOMPLETE = "private_recipient_source_index_hold_mapping_incomplete"
BLOCKED = "private_recipient_source_index_release_blocked_public_leak"

PRIVATE_REQUIRED = {
    "recipient_alias",
    "source_bundle_ref",
    "raw_source_url_private",
    "candidate_identity_private",
    "source_kind_label",
    "checked_date",
    "storage_location_private",
}
MAPPING_REQUIRED = {
    "capability_mapping",
    "public_card_alias",
    "public_card_source_bundle_ref",
    "privacy_review_label",
    "consent_or_terms_label",
    "owner_scope_label",
}
PUBLIC_LEAKS = {
    "public_raw_source_url",
    "public_candidate_identity",
    "public_contact_detail",
    "public_private_reply",
    "public_patient_material",
    "public_patient_sample",
    "public_treatment_or_trial_scope",
    "public_purchase_import_procurement",
}


def classify_private_index(row: dict[str, bool | str]) -> str:
    """Return the source-index readiness label for one candidate row."""
    if any(row.get(flag, False) for flag in PUBLIC_LEAKS):
        return BLOCKED
    if any(not row.get(flag, False) for flag in PRIVATE_REQUIRED):
        return MISSING_SOURCE
    if any(not row.get(flag, False) for flag in MAPPING_REQUIRED):
        return MAPPING_INCOMPLETE
    return READY


base_row = {flag: True for flag in PRIVATE_REQUIRED | MAPPING_REQUIRED}
base_row["recipient_alias"] = "recipient-candidate-001"
base_row["source_bundle_ref"] = "private-source-bundle-001"
base_row["storage_location_private"] = "private/recipient-source-index"

cases = {
    "ready": base_row,
    "missing_url": {**base_row, "raw_source_url_private": False},
    "missing_mapping": {**base_row, "capability_mapping": False},
    "public_url_leak": {**base_row, "public_raw_source_url": True},
    "public_trial_scope": {**base_row, "public_treatment_or_trial_scope": True},
}
expected = {
    "ready": READY,
    "missing_url": MISSING_SOURCE,
    "missing_mapping": MAPPING_INCOMPLETE,
    "public_url_leak": BLOCKED,
    "public_trial_scope": BLOCKED,
}
results = {name: classify_private_index(row) for name, row in cases.items()}
assert results == expected

decision_summary = {
    "current_state": "template_ready_no_real_sources",
    "ready_label": READY,
    "blocked_label": BLOCKED,
    "boundary": "real source rows stay ignored; public output uses labels",
}
decision_summary

## Decision

The public repo can carry the source-index template and labels. Completed source-index rows with real candidate URLs or identities must stay in ignored private storage.
